[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/chi_square_anova.ipynb)

# Chi-square tests and ANOVA full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the module's standard tests, several steps each), **tier 3** stretch problems (combine
ideas, spot the trap, work backwards) and **tier 4**, the case: all 96 trial batches of Tarnwick Components, a maker
of moulded power-tool housings (a fictional company; our own data), and the question of which of four resin
suppliers it should standardise on. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same tests with `scipy.stats`, `numpy` and
`pandas`, so you can run them on your own data. And a few cells you can move yourself: a fund's volatility sliding
along the chi-square curve, and three store layouts whose stores spread out while their means stay put.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Decimals as
decimals: `0.0263`, not `2.63%`; a p-value to four decimals is close enough. Numbers without thousands separators:
`60000`, not `60,000`. You can type the arithmetic too: `(58-50)**2/50`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). Pairs as names or numbers joined by
a dash, separated by commas: `1-2, 2-3` or `Alder-Cedar, Birch-Cedar`, in any order. You can also put your answer
between the quotes in a cell's last line, `practice("P1.5", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every test, formula and spreadsheet function from the course on two pages:
[chi-square and ANOVA cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/chi_square_anova_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, random, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the curve CHALK · rejection region VIOLET · the p-value GOLD; means SKY, between gaps VIOLET, within gaps MINT
GROUPS = [SKY, CORAL, MINT, VIOLET]

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(mpa|days?|minutes?|seconds?|tonnes?|kg)\b|°c|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def canon(words):
    """Pairs or names in any order: "Cedar-Alder, 2–1" and "1-2, Alder-Cedar" read the same."""
    s = words.lower().replace("–", "-").replace("—", "-").replace(" and ", ",")
    items = [i.strip() for i in s.split(",") if i.strip()]
    return ", ".join(sorted("-".join(sorted(p.strip() for p in i.split("-"))) for i in items))

def right(got, want, tol):
    if isinstance(want, str):
        return isinstance(got, str) and canon(got) == canon(want)
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Inferences about population variances',
     "q": 'Five lead times (days): 4, 7, 5, 8, 6. What is the sample variance s²?',
     "parts": ['s² (days²)'],
     "answer": [2.5], "tol": [0.0005], "function": 'VAR.S, DEVSQ',
     "working": 'x̄ = 6; squared deviations 4, 1, 1, 4, 0 add to 10; s² = 10/(5 − 1) = 2.5'},
    {"id": 'P1.2', "tier": 1, "group": 'Inferences about population variances',
     "q": 'n = 16, s = 5, and H0 says σ = 4. Compute the chi-square statistic and its degrees of freedom.',
     "parts": ['χ²', 'degrees of freedom'],
     "answer": [23.4375, 15], "tol": [0.005, 0], "function": '—',
     "working": 'χ² = (n − 1)s²/σ0² = 15 × 25/16 = 23.4375; df = 15'},
    {"id": 'P1.3', "tier": 1, "group": 'Inferences about population variances',
     "q": 'Two samples: s1 = 6 (n1 = 13), s2 = 4 (n2 = 16). Compute F = s1²/s2² and its two degrees of freedom.',
     "parts": ['F', 'numerator df', 'denominator df'],
     "answer": [2.25, 12, 15], "tol": [0.0005, 0, 0], "function": '—',
     "working": 'F = 36/16 = 2.25 with 12 and 15 degrees of freedom (numerator first)'},
    {"id": 'P1.4', "tier": 1, "group": 'Comparing three or more proportions',
     "q": 'Three regions each sent a renewal offer to 300 customers: 45, 60 and 75 renewed. Under H0 (equal renewal rates), what is the expected number of renewals in each region?',
     "parts": ['expected renewals in each region'],
     "answer": [60.0], "tol": [0.0005], "function": 'SUM',
     "working": 'pooled p̄ = 180/900 = 0.2; expected = 300 × 0.2 = 60'},
    {"id": 'P1.5', "tier": 1, "group": 'Independence and goodness of fit',
     "q": 'In a contingency table with n = 400, a row totals 120 and a column totals 80. What is the expected count in their cell under independence?',
     "parts": ['expected count'],
     "answer": [24.0], "tol": [0.0005], "function": '—',
     "working": 'row total × column total / n = 120 × 80/400 = 24'},
    {"id": 'P1.6', "tier": 1, "group": 'Independence and goodness of fit',
     "q": 'How many degrees of freedom does the chi-square test of independence have for a table with 4 rows and 3 columns?',
     "parts": ['degrees of freedom'],
     "answer": [6], "tol": [0], "function": '—',
     "working": '(r − 1)(c − 1) = 3 × 2 = 6'},
    {"id": 'P1.7', "tier": 1, "group": 'Independence and goodness of fit',
     "q": 'One cell has observed count 58 and expected count 50. What does it add to the chi-square statistic?',
     "parts": ['contribution to χ²'],
     "answer": [1.28], "tol": [0.0005], "function": '—',
     "working": '(f_o − f_e)²/f_e = 64/50 = 1.28'},
    {"id": 'P1.8', "tier": 1, "group": 'Experimental design and one-way ANOVA',
     "q": 'Three treatments, 6 units each, have means 10, 12 and 14 (overall mean 12). What is the between-treatments sum of squares, SSTR?',
     "parts": ['SSTR'],
     "answer": [48], "tol": [0.0005], "function": 'SUMPRODUCT',
     "working": 'SSTR = Σ n(x̄j − x̿)² = 6(4 + 0 + 4) = 48'},
    {"id": 'P1.9', "tier": 1, "group": 'Experimental design and one-way ANOVA',
     "q": 'An ANOVA with 3 treatments and 15 observations has MSTR = 40 and MSE = 8. Compute F and its p-value.',
     "parts": ['F', 'p-value'],
     "answer": [5.0, 0.026336095680589033], "tol": [0.0005, 5e-05], "function": 'F.DIST.RT',
     "working": 'F = 40/8 = 5.0 with 2 and 12 degrees of freedom; p = F.DIST.RT(5, 2, 12) ≈ 0.0263'},
    {"id": 'P1.10', "tier": 1, "group": 'Multiple comparisons, blocks and factorial experiments',
     "q": 'Four treatment means are compared two at a time, every pair at α = 0.05. How many comparisons are there, and what is the chance of at least one false alarm if all four means are equal (treat the tests as independent)?',
     "parts": ['number of comparisons', 'chance of at least one false alarm'],
     "answer": [6, 0.2649081093750002], "tol": [0, 0.0005], "function": '—',
     "working": 'C = 4 × 3/2 = 6; 1 − 0.95⁶ ≈ 0.265'},
    {"id": 'P2.1', "tier": 2, "group": 'Inferences about population variances',
     "q": 'Twenty deliveries from a new carrier: the standard deviation of delivery time is s = 1.8 days. Give the 95% confidence interval for σ.',
     "parts": ['95% interval for σ: lower end (days)', '95% interval for σ: upper end (days)'],
     "answer": [1.368882709793774, 2.629028945014594], "tol": [0.0005, 0.0005], "function": 'CHISQ.INV.RT, CHISQ.INV, SQRT',
     "working": 'df 19; CHISQ.INV.RT(0.025, 19) ≈ 32.852, CHISQ.INV(0.025, 19) ≈ 8.907; σ² from 19 × 3.24/32.852 ≈ 1.874 to 19 × 3.24/8.907 ≈ 6.912; σ from ≈ 1.369 to ≈ 2.629 days'},
    {"id": 'P2.2', "tier": 2, "group": 'Inferences about population variances',
     "q": "A reactor's temperature must have σ ≤ 0.5 °C. Thirty readings give s = 0.62 °C. Test at α = 0.05: the chi-square statistic and the p-value.",
     "parts": ['χ²', 'p-value'],
     "answer": [44.5904, 0.032232874538981826], "tol": [0.005, 0.0005], "function": 'CHISQ.DIST.RT',
     "working": 'H0 σ² ≤ 0.25, Ha σ² > 0.25; χ² = 29 × 0.3844/0.25 ≈ 44.59, df 29; p = CHISQ.DIST.RT(44.59, 29) ≈ 0.032: reject; the temperature varies more than allowed'},
    {"id": 'P2.3', "tier": 2, "group": 'Inferences about population variances',
     "q": "Two analysts' monthly forecast errors (%): the first, n = 16, s = 4.1; the second, n = 13, s = 2.6. Is one less consistent? Two-tailed F test at α = 0.05: F and p.",
     "parts": ['F (larger variance on top)', 'p-value (two-tailed)'],
     "answer": [2.486686390532544, 0.11861600006364847], "tol": [0.005, 0.0005], "function": 'F.DIST.RT, F.TEST',
     "working": 'larger variance on top: F = 16.81/6.76 ≈ 2.487 with 15 and 12 df; p = 2 × F.DIST.RT(2.487, 15, 12) ≈ 0.119: do not reject'},
    {"id": 'P2.4', "tier": 2, "group": 'Comparing three or more proportions',
     "q": 'Trial users who upgraded to a paid plan, in three markets: 48 of 400, 70 of 350, 54 of 450. Test whether the three upgrade rates are equal, at α = 0.05: χ², df, p.',
     "parts": ['χ²', 'degrees of freedom', 'p-value'],
     "answer": [12.921907519681477, 2, 0.0015633039825747597], "tol": [0.005, 0, 0.0005], "function": 'CHISQ.TEST',
     "working": 'p̄ = 172/1200 ≈ 0.1433; expected upgrades 57.33, 50.17, 64.5; χ² ≈ 12.92, df 2; p ≈ 0.0016: reject'},
    {"id": 'P2.5', "tier": 2, "group": 'Comparing three or more proportions',
     "q": 'For P2.4, use the Marascuilo procedure: which pairs of markets differ? (List the pairs, e.g. 1–2.)',
     "parts": ['the pairs that differ, smallest first, e.g. 1–2, 2–3 (a hyphen is fine)'],
     "answer": ['1–2, 2–3'], "tol": [None], "function": 'CHISQ.INV.RT, SQRT',
     "working": 'rates 0.12, 0.20, 0.12; √χ²(0.05, 2) = √5.991 ≈ 2.448; 1–2: 0.08 vs CV ≈ 0.0657 (differs); 1–3: 0 vs ≈ 0.0547; 2–3: 0.08 vs ≈ 0.0644 (differs). The second market differs from both others'},
    {"id": 'P2.6', "tier": 2, "group": 'Independence and goodness of fit',
     "q": 'Orders by payment method (rows: card, wallet) and basket size (columns: small, medium, large): card 90, 110, 100; wallet 110, 70, 20. Are payment method and basket size independent? χ², df, p.',
     "parts": ['χ²', 'degrees of freedom', 'p-value'],
     "answer": [46.06481481481481, 2, 9.934653100174273e-11], "tol": [0.005, 0, 1e-09], "function": 'CHISQ.TEST',
     "working": 'expected: card 120, 108, 72; wallet 80, 72, 48; χ² ≈ 46.06, df 2; p < 0.0001: reject; wallet orders lean small'},
    {"id": 'P2.7', "tier": 2, "group": 'Independence and goodness of fit',
     "q": "Last year's returns split by reason: wrong size 40%, damaged 15%, not as described 25%, changed mind 20%. This quarter's 300 returns: 108, 60, 72, 60. Has the mix changed? χ², df, p at α = 0.05.",
     "parts": ['χ²', 'degrees of freedom', 'p-value'],
     "answer": [6.32, 3, 0.09703806460956199], "tol": [0.005, 0, 0.0005], "function": 'CHISQ.TEST',
     "working": 'expected 120, 45, 75, 60; χ² = 1.2 + 5 + 0.12 + 0 = 6.32, df 3; p ≈ 0.097: do not reject (damaged returns are up, but not clearly)'},
    {"id": 'P2.8', "tier": 2, "group": 'Experimental design and one-way ANOVA',
     "q": 'Page-load times (seconds) through three content networks, five tests each: first 1.8, 2.1, 1.9, 2.3, 2.0; second 2.4, 2.2, 2.6, 2.5, 2.3; third 1.9, 2.0, 2.2, 1.8, 2.1. One-way ANOVA at α = 0.05: F and p.',
     "parts": ['F', 'p-value'],
     "answer": [8.75862068965516, 0.004514741360710062], "tol": [0.0005, 5e-05], "function": 'AVERAGE, DEVSQ, F.DIST.RT',
     "working": 'means 2.02, 2.40, 2.00; SSTR ≈ 0.508, SSE ≈ 0.348; MSTR ≈ 0.254, MSE ≈ 0.029; F ≈ 8.76 (2, 12); p ≈ 0.0045: reject'},
    {"id": 'P2.9', "tier": 2, "group": 'Experimental design and one-way ANOVA',
     "q": 'Average order value ($) in four regions, 8 stores each: means 120, 126, 131, 123; standard deviations 9, 8, 10, 7. One-way ANOVA from these summaries at α = 0.05: F and p.',
     "parts": ['F', 'p-value'],
     "answer": [2.3945578231292517, 0.0894685304253463], "tol": [0.005, 0.0005], "function": 'F.DIST.RT',
     "working": 'grand mean 125; SSTR = 8(25 + 1 + 36 + 4) = 528; SSE = 7(81 + 64 + 100 + 49) = 2058; F = 176/73.5 ≈ 2.39 (3, 28); p ≈ 0.089: do not reject'},
    {"id": 'P2.10', "tier": 2, "group": 'Multiple comparisons, blocks and factorial experiments',
     "q": "For P2.8, compute Fisher's LSD at α = 0.05 and say which pairs of networks differ.",
     "parts": ['LSD (seconds)', 'the pairs that differ, smallest first, e.g. 1–2 (a hyphen is fine)'],
     "answer": [0.23466532343314161, '1–2, 2–3'], "tol": [0.0005, None], "function": 'T.INV.2T, SQRT',
     "working": 'LSD = T.INV.2T(0.05, 12) × √(0.029 × 2/5) ≈ 2.179 × 0.1077 ≈ 0.235; differences 0.38, 0.02, 0.40: the second network differs from both'},
    {"id": 'P3.1', "tier": 3, "group": 'Inferences about population variances',
     "q": "A fund's mandate says its monthly volatility σ is at most 4%. With 24 monthly returns and α = 0.05, how large must the sample standard deviation be before the test rejects the mandate? (Work backwards from the critical value.)",
     "parts": ['the smallest s that rejects (%)'],
     "answer": [4.946493646274267], "tol": [0.0005], "function": 'CHISQ.INV.RT, SQRT',
     "working": 'reject if (n − 1)s²/16 ≥ CHISQ.INV.RT(0.05, 23) ≈ 35.172; s ≥ √(35.172 × 16/23) ≈ 4.946%: a sample s of 4.8% is not enough'},
    {"id": 'P3.2', "tier": 3, "group": 'Inferences about population variances',
     "q": "A colleague tests whether two machines differ in spread (two-tailed). Old: n = 16, s = 4.2; new: n = 21, s = 2.5. They compute F = s_new²/s_old² ≈ 0.354 and report F.DIST.RT(0.354, 20, 15) ≈ 0.98: 'no difference'. What is the right p-value?",
     "parts": ['the right p-value (two-tailed)'],
     "answer": [0.03184777833196195], "tol": [0.0005], "function": 'F.DIST.RT, F.TEST',
     "working": 'Put the larger variance on top: F = 17.64/6.25 ≈ 2.822 with 15 and 20 df; two-tailed p = 2 × F.DIST.RT(2.822, 15, 20) ≈ 0.032 (= F.TEST on the raw data): the new machine is more consistent'},
    {"id": 'P3.3', "tier": 3, "group": 'Independence and goodness of fit',
     "q": 'To check whether 150 daily price changes are normal, an analyst estimates μ and σ from the data, builds 6 equal-probability bins (25 expected in each) and counts 31, 22, 18, 20, 24, 35. Compute χ² and the p-value with the right degrees of freedom, and the p-value CHISQ.TEST would report. Which conclusion is right?',
     "parts": ['χ²', 'p-value with the right df', 'the p-value CHISQ.TEST reports'],
     "answer": [8.8, 0.03207164082526537, 0.11731235792385697], "tol": [0.0005, 0.0005, 0.0005], "function": 'CHISQ.DIST.RT, CHISQ.TEST',
     "working": 'χ² = (36 + 9 + 49 + 25 + 1 + 100)/25 = 8.8; df = 6 − 1 − 2 = 3 (two parameters estimated): p = CHISQ.DIST.RT(8.8, 3) ≈ 0.032, reject normality (fat tails). CHISQ.TEST uses df 5: p ≈ 0.117, the wrong answer'},
    {"id": 'P3.4', "tier": 3, "group": 'Independence and goodness of fit',
     "q": 'Defects by type are expected to split 60%, 25%, 10%, 4% and 1%. Of 150 defects, 82, 41, 15, 8 and 4 fell in the five types. Check the expected counts, combine where needed, then test the mix: χ², df, p.',
     "parts": ['χ² (after combining)', 'degrees of freedom', 'p-value'],
     "answer": [3.737777777777778, 3, 0.2912070465928285], "tol": [0.005, 0, 0.0005], "function": 'CHISQ.DIST.RT',
     "working": 'expected 90, 37.5, 15, 6, 1.5: the last is below 5, so combine the last two (12 observed, 7.5 expected); χ² = 0.711 + 0.327 + 0 + 2.7 ≈ 3.738, df 3; p ≈ 0.291: do not reject'},
    {"id": 'P3.5', "tier": 3, "group": 'Experimental design and one-way ANOVA',
     "q": 'A one-way ANOVA (3 treatments, 6 units each) gives F = 6.0 and p ≈ 0.012. Suppose every observation sat 1.5 times as far from its own treatment mean, with the same treatment means. What would F and p be?',
     "parts": ['F', 'p-value'],
     "answer": [2.6666666666666665, 0.10212218320458177], "tol": [0.0005, 0.0005], "function": 'F.DIST.RT',
     "working": 'SSE and MSE grow by 1.5² = 2.25 while MSTR stays: F = 6/2.25 ≈ 2.667 (2, 15); p ≈ 0.102: no longer significant'},
    {"id": 'P3.6', "tier": 3, "group": 'Multiple comparisons, blocks and factorial experiments',
     "q": 'Four treatments give six pairwise LSD p-values: 0.004, 0.012, 0.021, 0.034, 0.18, 0.47. How many pairs differ at α = 0.05 per comparison, and how many with the Bonferroni adjustment?',
     "parts": ['pairs that differ at 0.05', 'pairs that differ with Bonferroni'],
     "answer": [4, 1], "tol": [0, 0], "function": '—',
     "working": 'per comparison: 4 below 0.05; Bonferroni: α/C = 0.05/6 ≈ 0.0083, so only 0.004 survives: 1 pair'},
    {"id": 'P3.7', "tier": 3, "group": 'Multiple comparisons, blocks and factorial experiments',
     "q": 'Five claims handlers each processed a test claim with three software tools (minutes): 13.5, 13, 15.5; 14.5, 17, 19.5; 18.5, 21, 20.5; 22.5, 23, 23.5; 23.5, 26, 28.5 (one handler per group, tools in order). Analyse it as a completely randomized design, then as a randomized block design with handlers as blocks: the two p-values.',
     "parts": ['p-value, completely randomized', 'p-value, randomized block'],
     "answer": [0.6289207361954299, 0.008963271594131794], "tol": [0.0005, 0.0005], "function": 'DEVSQ, F.DIST.RT',
     "working": 'CRD: F ≈ 0.48 (2, 12), p ≈ 0.63. Blocks: SSTR 22.5, SSBL 270, SSE 10; F = 11.25/1.25 = 9.0 (2, 8), p ≈ 0.009. The handlers differ by far more than the tools; blocking removes that spread'},
    {"id": 'P3.8', "tier": 3, "group": 'Multiple comparisons, blocks and factorial experiments',
     "q": 'A 2 × 2 experiment: packaging (standard, eco) × shipping (standard, express), three regions per combination; customer rating out of 10. Standard/standard 7.6, 7.9, 7.3; standard/express 8.4, 8.1, 8.7; eco/standard 7.8, 7.4, 7.7; eco/express 9.6, 9.9, 9.3. Test the interaction: F and p.',
     "parts": ['F for the interaction', 'p-value'],
     "answer": [13.031914893617055, 0.006881931605808744], "tol": [0.005, 0.0005], "function": 'AVERAGE, DEVSQ, F.DIST.RT',
     "working": 'cell means 7.6, 8.4, 7.63, 9.6; SSAB ≈ 1.021, SSE ≈ 0.627 (df 8, MSE ≈ 0.078); F ≈ 13.0 (1, 8); p ≈ 0.007: interaction. Express lifts ratings far more with eco packaging'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": "Reproduce the case: each supplier's mean strength and standard deviation (MPa).",
     "parts": ['Alder: mean (MPa)', 'Birch: mean (MPa)', 'Cedar: mean (MPa)', 'Dunmore: mean (MPa)', 'Alder: s (MPa)', 'Birch: s (MPa)', 'Cedar: s (MPa)', 'Dunmore: s (MPa)'],
     "answer": [68.0, 66.4, 69.01, 64.4, 2.4, 2.5, 5.0, 2.5], "tol": [0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005], "function": 'AVERAGEIF, STDEV.S',
     "working": 'AVERAGEIF and STDEV.S by supplier: means 68.00, 66.40, 69.01, 64.40; s 2.40, 2.50, 5.00, 2.50'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": "Rejects: test whether the four suppliers' reject rates are equal. Total rejects, χ² and p.",
     "parts": ['total rejects, all four', 'χ²', 'p-value'],
     "answer": [1187, 49.94529939884678, 8.206380019970538e-11], "tol": [0, 0.005, 1e-09], "function": 'SUMIF, CHISQ.TEST',
     "working": 'rejects 256, 256, 400, 275 of 12,000 each; p̄ ≈ 0.0247; χ² ≈ 49.95, df 3; p < 0.0001'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": 'Marascuilo on the reject rates: which supplier pairs differ?',
     "parts": ["the pairs that differ, in the suppliers' order, e.g. Alder–Birch (a hyphen is fine)"],
     "answer": ['Alder–Cedar, Birch–Cedar, Cedar–Dunmore'], "tol": [None], "function": 'CHISQ.INV.RT, SQRT',
     "working": "√χ²(0.05, 3) = √7.815 ≈ 2.796; Cedar's 3.33% differs from all three; the others (2.13%, 2.13%, 2.29%) don't"},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": "Consistency: for each supplier, test H0: σ ≤ 3 MPa against Ha: σ > 3 MPa (α = 0.05). Give Cedar's χ² and p, and say which suppliers fail.",
     "parts": ['Cedar: χ²', 'Cedar: p-value', 'the suppliers that fail'],
     "answer": [63.81092592592594, 1.055091678758835e-05, 'Cedar'], "tol": [0.05, 1e-06, None], "function": 'STDEV.S, CHISQ.DIST.RT',
     "working": 'χ² = 23 s²/9: Alder ≈ 14.7, Birch ≈ 16.0, Cedar ≈ 63.8, Dunmore ≈ 16.0; p ≈ 0.90, 0.86, 0.00001, 0.85: only Cedar fails'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": 'One-way ANOVA on mean strength for the three consistent suppliers (Alder, Birch, Dunmore): F, the degrees of freedom and p.',
     "parts": ['F', 'numerator df', 'denominator df', 'p-value'],
     "answer": [12.781033568948434, 2, 69, 1.8969618690412027e-05], "tol": [0.005, 0, 0, 1e-06], "function": 'DEVSQ, F.DIST.RT',
     "working": 'means 68.00, 66.40, 64.40; MSE ≈ 6.10; F ≈ 12.78 (2, 69); p ≈ 0.00002'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": 'Pairwise LSD p-values for the three, and which pairs survive the Bonferroni adjustment (3 comparisons).',
     "parts": ['Alder–Birch: p-value', 'Alder–Dunmore: p-value', 'Birch–Dunmore: p-value', 'the pairs that survive Bonferroni (a hyphen is fine)'],
     "answer": [0.02837637001093407, 3.5111744232096953e-06, 0.0065098367882181485, 'Alder–Dunmore, Birch–Dunmore'], "tol": [0.0005, 1e-05, 0.0005, None], "function": 'T.DIST.2T, SQRT',
     "working": "Alder–Birch p ≈ 0.028, Alder–Dunmore p ≈ 0.000004, Birch–Dunmore p ≈ 0.0065; α/3 ≈ 0.0167: Alder–Birch does not survive, so the two aren't shown to differ"},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": "Go further: run the ANOVA with all four suppliers. What are F and p, and why is this test less trustworthy than the three-supplier one? (Give the ratio of Cedar's variance to Alder's.)",
     "parts": ['F (all four)', 'p-value', "Cedar's variance ÷ Alder's"],
     "answer": [8.977621711435173, 2.8196904207189455e-05, 4.334013577634326], "tol": [0.005, 1e-06, 0.005], "function": 'F.DIST.RT, F.TEST',
     "working": "F ≈ 8.98 (3, 92), p ≈ 0.00003; but ANOVA assumes equal variances and Cedar's is ≈ 4.3 times Alder's (F test p ≈ 0.0008), so check spreads first and judge Cedar on consistency"},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Tarnwick Components',
     "q": "Write the recommendation. Anchor it: the yearly resin saving of Birch over Alder (400 tonnes a year), and the share of Cedar batches a normal model with Cedar's mean and spread expects below 62 MPa.",
     "parts": ['yearly saving of Birch over Alder ($)', 'share of Cedar batches below 62 MPa (a decimal)'],
     "answer": [59999.99999999996, 0.08037961285737212], "tol": [0.5, 0.0005], "function": 'NORM.S.DIST',
     "working": '($2.35 − $2.20) × 400,000 kg = $60,000 a year; NORM.S.DIST((62 − 69.01)/5.00, TRUE) ≈ 0.080: about one Cedar batch in twelve below spec (2 of 24 in the trial). Recommendation: standardise on Birch; keep Alder as second source; recheck the spread every quarter'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def split(text, q):
    """The typed answer, one part per comma; when the last part is words (a list of pairs), it keeps its commas."""
    pieces = [p for p in re.split(r"[,;]", text) if p.strip()]
    n = len(q["answer"])
    if isinstance(q["answer"][-1], str):
        nums, words = pieces[:n - 1], pieces[n - 1:]
        return [read_part(p) for p in nums] + ([", ".join(w.strip() for w in words)] if words else [])
    return [read_part(p) for p in pieces]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = split(box.value, q)
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

## Tier 1 · Warm-up

One idea, one step, small numbers: a sample variance, a chi-square statistic for one variance, an F ratio for two,
expected counts, degrees of freedom, one cell's contribution to χ², the between-treatments sum of squares, an F test
from its mean squares, and the chance of a false alarm when you make many comparisons.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

`scipy.stats.chi2` is the chi-square curve and `scipy.stats.f` the F curve. For either, `sf(x, df)` is the area to
the right of x (a spreadsheet's CHISQ.DIST.RT and F.DIST.RT), `cdf` the area to the left (CHISQ.DIST with TRUE),
`isf(p, df)` the value with area p to its right (CHISQ.INV.RT, F.INV.RT) and `ppf(p, df)` the value with area p to
its left (CHISQ.INV). The rest is arithmetic. Each line below computes one warm-up answer and checks it against the
practice set.

In [ ]:
from scipy.stats import chi2, f as fdist

lead = [4, 7, 5, 8, 6]
s2 = np.var(lead, ddof=1)                                       # P1.1: ddof=1 divides by n − 1 (VAR.S)
x2, df = 15 * 5 ** 2 / 4 ** 2, 16 - 1                           # P1.2
F, df1, df2 = 6 ** 2 / 4 ** 2, 13 - 1, 16 - 1                   # P1.3: numerator df first
pbar = (45 + 60 + 75) / 900; e_renew = 300 * pbar               # P1.4
e_cell = 120 * 80 / 400                                         # P1.5
df_tab = (4 - 1) * (3 - 1)                                      # P1.6
contrib = (58 - 50) ** 2 / 50                                   # P1.7
means = np.array([10, 12, 14])
sstr = 6 * ((means - means.mean()) ** 2).sum()                  # P1.8
F9 = 40 / 8; p9 = fdist.sf(F9, 3 - 1, 15 - 3)                   # P1.9
C = math.comb(4, 2); alarm = 1 - 0.95 ** C                      # P1.10

print(f"s² {s2}   χ² {x2} (df {df})   F {F} ({df1}, {df2})   expected {e_renew:g}, {e_cell:g}   df {df_tab}")
print(f"contribution {contrib}   SSTR {sstr:g}   F {F9}, p {p9:.4f}   {C} comparisons, P(false alarm) {alarm:.3f}")
for k, v in (("P1.1", [s2]), ("P1.2", [x2, df]), ("P1.3", [F, df1, df2]), ("P1.4", [e_renew]), ("P1.5", [e_cell]),
             ("P1.6", [df_tab]), ("P1.7", [contrib]), ("P1.8", [sstr]), ("P1.9", [F9, p9]), ("P1.10", [C, alarm])):
    assert agrees(k, v), k

### Slide the volatility: watch the p-value

The course's fund: its mandate says the monthly volatility σ is at most 4%. Twenty-four monthly returns give
s = 4.8%. The test statistic is (n − 1)s²/σ₀² = 23 × 23.04/16 = 33.12, on 23 degrees of freedom, and the p-value is
the gold area to its right: about 0.079, so we can't reject the mandate. The violet region is where the test would
reject, beyond the critical value 35.172. Slide s up and watch χ² cross into it; change n and watch the curve move.

In [ ]:
def variance_picture(s=4.8, n=24):
    df = n - 1
    x2 = df * s ** 2 / 4 ** 2
    p, crit = chi2.sf(x2, df), chi2.isf(0.05, df)
    xs = np.linspace(0.01, max(3 * df, x2 + 10), 500)
    fig, ax = plt.subplots(figsize=(11.5, 4))
    fig.subplots_adjust(left=0.05, right=0.98, top=0.8, bottom=0.18)
    ax.plot(xs, chi2.pdf(xs, df), color=CHALK)
    tail = xs[xs >= x2]; ax.fill_between(tail, chi2.pdf(tail, df), color=GOLD, alpha=0.8, lw=0)
    rr = xs[xs >= crit]; ax.fill_between(rr, chi2.pdf(rr, df), color=VIOLET, alpha=0.55, lw=0, hatch="//",
                                         edgecolor=VIOLET)
    ax.axvline(x2, color=GOLD, lw=1.6)
    ax.axvline(crit, color=VIOLET, ls="--", lw=1.4)
    ax.text(crit, chi2.pdf(df - 2 if df > 2 else 1, df) * 0.9, f" critical {crit:.3f}", color=VIOLET, fontsize=12)
    ax.set_xlim(0, xs[-1]); ax.set_ylim(0, None); ax.set_yticks([]); ax.set_xlabel(f"χ² with {df} degrees of freedom")
    verdict = "reject: σ above 4%" if p <= 0.05 else "do not reject the mandate"
    fig.suptitle(f"s = {s:.2f}%, n = {n}: χ² = {x2:.2f}, p = {p:.4f} (critical {crit:.3f}) → {verdict}", x=0.05,
                 ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert round(23 * 4.8 ** 2 / 16, 2) == 33.12 and round(chi2.sf(33.12, 23), 3) == 0.079   # the video's numbers
assert round(chi2.isf(0.05, 23), 3) == 35.172 and round(chi2.isf(0.025, 23), 3) == 38.076 and round(chi2.ppf(0.025, 23), 3) == 11.689

In [ ]:
interact(variance_picture, s=FloatSlider(value=4.8, min=3.0, max=7.0, step=0.05, description="s (%)"),
         n=IntSlider(value=24, min=6, max=60, step=1, description="n"));

## Tier 2 · Core

The module's standard tests, several steps each: an interval for σ, a test of a variance against a limit, the
two-tailed F test, equal proportions with Marascuilo's follow-up, a test of independence, a goodness of fit to last
year's mix, a one-way ANOVA from raw data and from summaries, and Fisher's LSD.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Variances.** An interval for σ is two lines with `chi2.isf` and `chi2.ppf`; a test is `chi2.sf` of
(n − 1)s²/σ₀². For two variances, put the larger on top and double `f.sf` for two tails.

**Counts.** `stats.chi2_contingency(table)` takes a table of observed counts and returns the statistic, the p-value,
the degrees of freedom and the expected counts: it does independence and equal proportions alike. For a goodness of
fit, `stats.chisquare(observed, expected)`; its `ddof` argument takes off one degree of freedom per estimated
parameter.

**ANOVA.** `stats.f_oneway(group1, group2, group3)` returns F and the p-value from raw data. From summaries, build
the sums of squares yourself: SSTR = Σ n(x̄ − x̿)², SSE = Σ(n − 1)s².

In [ ]:
n, s = 20, 1.8                                                                          # P2.1
lo, hi = math.sqrt((n - 1) * s ** 2 / chi2.isf(0.025, n - 1)), math.sqrt((n - 1) * s ** 2 / chi2.ppf(0.025, n - 1))
assert agrees("P2.1", [lo, hi])
x2 = 29 * 0.62 ** 2 / 0.5 ** 2                                                          # P2.2: upper tail
assert agrees("P2.2", [x2, chi2.sf(x2, 29)])
F = 4.1 ** 2 / 2.6 ** 2                                                                 # P2.3: larger on top, two tails
assert agrees("P2.3", [F, 2 * fdist.sf(F, 15, 12)])
print(f"P2.1 σ from {lo:.3f} to {hi:.3f} days · P2.2 χ² {x2:.2f}, p {chi2.sf(x2, 29):.4f} · "
      f"P2.3 F {F:.3f}, p {2 * fdist.sf(F, 15, 12):.4f}")

upgraded, users = np.array([48, 70, 54]), np.array([400, 350, 450])                    # P2.4: equal proportions
r = stats.chi2_contingency([upgraded, users - upgraded])
print("P2.4:", round(r.statistic, 2), "df", r.dof, "p", round(r.pvalue, 4))
assert agrees("P2.4", [r.statistic, r.dof, r.pvalue])

def marascuilo(x, n, names, alpha=0.05):
    """Every pair's |p̄ᵢ − p̄ⱼ| against its critical value √χ²(α, k − 1) × √(p̄ᵢq̄ᵢ/nᵢ + p̄ⱼq̄ⱼ/nⱼ)."""
    p = np.array(x) / np.array(n); root = math.sqrt(chi2.isf(alpha, len(x) - 1)); out = []
    for i in range(len(x)):
        for j in range(i + 1, len(x)):
            cv = root * math.sqrt(p[i] * (1 - p[i]) / n[i] + p[j] * (1 - p[j]) / n[j])
            out.append((f"{names[i]}–{names[j]}", abs(p[i] - p[j]), cv, abs(p[i] - p[j]) > cv))
    return out

m = marascuilo(upgraded, users, ["1", "2", "3"])                                        # P2.5
for pair, d, cv, differs in m: print(f"  markets {pair}: |difference| {d:.4f} vs {cv:.4f}" + ("  differs" if differs else ""))
assert agrees("P2.5", [", ".join(pair for pair, *_, differs in m if differs)])

r = stats.chi2_contingency([[90, 110, 100], [110, 70, 20]])                             # P2.6: independence
assert agrees("P2.6", [r.statistic, r.dof, r.pvalue])
obs = [108, 60, 72, 60]                                                                 # P2.7: goodness of fit
r = stats.chisquare(obs, [300 * s for s in (0.40, 0.15, 0.25, 0.20)])
assert agrees("P2.7", [r.statistic, len(obs) - 1, r.pvalue])
print(f"P2.6 χ² {stats.chi2_contingency([[90, 110, 100], [110, 70, 20]]).statistic:.2f} · P2.7 χ² {r.statistic:.2f}, p {r.pvalue:.3f}")

cdn = [[1.8, 2.1, 1.9, 2.3, 2.0], [2.4, 2.2, 2.6, 2.5, 2.3], [1.9, 2.0, 2.2, 1.8, 2.1]]     # P2.8: ANOVA, raw data
r = stats.f_oneway(*cdn)
assert agrees("P2.8", [r.statistic, r.pvalue])
ns, ms, ss = np.array([8] * 4), np.array([120, 126, 131, 123]), np.array([9, 8, 10, 7])    # P2.9: from summaries
grand = (ns * ms).sum() / ns.sum()
sstr, sse = (ns * (ms - grand) ** 2).sum(), ((ns - 1) * ss ** 2).sum()
F = (sstr / 3) / (sse / (ns.sum() - 4))
assert agrees("P2.9", [F, fdist.sf(F, 3, ns.sum() - 4)])
mse = sum(np.var(g, ddof=1) * (len(g) - 1) for g in cdn) / (15 - 3)                     # P2.10: Fisher's LSD
lsd = stats.t.ppf(0.975, 12) * math.sqrt(mse * (1 / 5 + 1 / 5))
gm = [np.mean(g) for g in cdn]
pairs = ", ".join(f"{i + 1}–{j + 1}" for i in range(3) for j in range(i + 1, 3) if abs(gm[i] - gm[j]) > lsd)
print(f"P2.8 F {r.statistic:.2f}, p {r.pvalue:.4f} · P2.9 F {F:.2f} · P2.10 LSD {lsd:.3f}: pairs {pairs}")
assert agrees("P2.10", [lsd, pairs])

In [ ]:
# The test of independence in one call: the video's 600 customers, channel × plan
table = [[92, 64, 44],      # search:   basic, standard, premium
         [93, 39, 18],      # social media
         [115, 77, 58]]     # referral
stats.chi2_contingency(table)

In [ ]:
r = stats.chi2_contingency(table)
assert round(r.statistic, 2) == 13.36 and round(r.pvalue, 4) == 0.0096 and r.dof == 4
assert r.expected_freq.round(6).tolist() == [[100, 60, 40], [75, 45, 30], [125, 75, 50]]

In [ ]:
# One-way ANOVA in one call: the video's three store layouts (weekly sales, $ thousands)
layout1, layout2, layout3 = [37, 40, 41, 42, 45], [39, 44, 45, 45, 47], [43, 45, 48, 48, 51]
stats.f_oneway(layout1, layout2, layout3)

In [ ]:
r = stats.f_oneway(layout1, layout2, layout3)
assert round(r.statistic, 6) == 5.0 and round(r.pvalue, 4) == 0.0263

### The full ANOVA table, one line per row

`f_oneway` gives F and p. The table behind them is a few lines of pandas: each layout's mean and sum of squared
deviations, then the two sums of squares, their degrees of freedom and mean squares.

In [ ]:
def anova_table(groups):
    """The one-way ANOVA table from a dict {name: values}."""
    allv = np.concatenate([np.asarray(v, float) for v in groups.values()])
    grand, k, nt = allv.mean(), len(groups), len(allv)
    sstr = sum(len(v) * (np.mean(v) - grand) ** 2 for v in groups.values())
    sse = sum(((np.asarray(v) - np.mean(v)) ** 2).sum() for v in groups.values())
    t = pd.DataFrame({"sum of squares": [sstr, sse, sstr + sse], "df": [k - 1, nt - k, nt - 1]},
                     index=["treatments", "error", "total"])
    t["mean square"] = t["sum of squares"] / t["df"]; t.loc["total", "mean square"] = np.nan
    t["F"] = [t.loc["treatments", "mean square"] / t.loc["error", "mean square"], np.nan, np.nan]
    t["p-value"] = [fdist.sf(t.iloc[0]["F"], k - 1, nt - k), np.nan, np.nan]
    return t

layouts = {"layout 1": layout1, "layout 2": layout2, "layout 3": layout3}
print(pd.DataFrame({k: [np.mean(v), ((np.array(v) - np.mean(v)) ** 2).sum()] for k, v in layouts.items()},
                   index=["mean", "DEVSQ"]).to_string(), "\n")
t = anova_table(layouts)
print(t.round(4).to_string())
assert t.loc["treatments", "sum of squares"] == 90 and t.loc["error", "sum of squares"] == 108 and t.loc["treatments", "F"] == 5

### Spread the stores out: the gap against the spread

The same three layouts. Slide the spread: every store moves further from its own layout's mean, by the same factor,
and the means don't move at all. The between-layouts sum of squares stays at 90; the within-layouts one grows with
the square of the factor, and F falls. At twice the spread, F is 1.25 and p is about 0.32: the same gaps between the
means now mean nothing.

In [ ]:
def spread_picture(factor=1.0):
    groups = [[np.mean(g) + factor * (v - np.mean(g)) for v in g] for g in (layout1, layout2, layout3)]
    r = stats.f_oneway(*groups)
    fig, ax = plt.subplots(figsize=(11.5, 4.4))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.14)
    ax.axhline(44, color=DIM, ls="--", lw=1.4)
    for j, g in enumerate(groups):
        ax.scatter([j] * len(g), g, s=90, color=GROUPS[j], zorder=3)
        ax.plot([j - 0.25, j + 0.25], [np.mean(g)] * 2, color=SKY, lw=3)
    ax.set_xticks([0, 1, 2], ["layout 1", "layout 2", "layout 3"]); ax.set_xlim(-0.6, 2.6); ax.set_ylim(25, 63)
    ax.set_ylabel("weekly sales ($ thousands)")
    fig.suptitle(f"spread × {factor:.2f}: F = {r.statistic:.2f}, p = {r.pvalue:.3f}", x=0.07, ha="left", fontsize=15,
                 color=GOLD)
    footnote(fig); plt.show()

r2 = stats.f_oneway(*[[np.mean(g) + 2 * (v - np.mean(g)) for v in g] for g in (layout1, layout2, layout3)])
assert round(r2.statistic, 6) == 1.25 and round(r2.pvalue, 2) == 0.32                      # the video's numbers

In [ ]:
interact(spread_picture, factor=FloatSlider(value=1.0, min=0.5, max=3.0, step=0.05, description="spread ×"));

## Tier 3 · Stretch

Combine ideas, choose the method, spot the trap, work backwards: the sample spread that would break a mandate, an F
ratio upside down, a goodness of fit that estimated its parameters, expected counts too small to use, the same means
with more noise, many comparisons, blocking, and an interaction.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**The traps.** `stats.chisquare(observed, expected, ddof=2)` takes off the two parameters you estimated (a
spreadsheet's CHISQ.TEST can't). Combine categories before testing when an expected count is under five.

**Blocks and factorials by hand.** Put the data in a pandas DataFrame with one row per observation and a column for
each factor; then every sum of squares is a `groupby` away: the treatment means, the block means (or the other
factor's), and the cell means. SST = SSTR + SSBL + SSE for blocks; SST = SSA + SSB + SSAB + SSE for a factorial.

In [ ]:
s_min = math.sqrt(chi2.isf(0.05, 23) * 4 ** 2 / 23)                                    # P3.1: work backwards
assert agrees("P3.1", [s_min])
F = 4.2 ** 2 / 2.5 ** 2                                                                 # P3.2: larger on top
assert agrees("P3.2", [2 * fdist.sf(F, 15, 20)])
bins = [31, 22, 18, 20, 24, 35]                                                         # P3.3: estimated μ and σ
right_df, chisq_test = stats.chisquare(bins, [25] * 6, ddof=2), stats.chisquare(bins, [25] * 6)
print(f"P3.1 s ≥ {s_min:.3f}% · P3.2 p {2 * fdist.sf(F, 15, 20):.3f} · P3.3 p {right_df.pvalue:.3f} with df 3, "
      f"{chisq_test.pvalue:.3f} with df 5 (wrong)")
assert agrees("P3.3", [right_df.statistic, right_df.pvalue, chisq_test.pvalue])
obs, exp = np.array([82, 41, 15, 8, 4]), 150 * np.array([0.60, 0.25, 0.10, 0.04, 0.01])  # P3.4: combine the last two
obs2, exp2 = np.append(obs[:3], obs[3:].sum()), np.append(exp[:3], exp[3:].sum())
r = stats.chisquare(obs2, exp2)
assert agrees("P3.4", [r.statistic, len(obs2) - 1, r.pvalue])
assert agrees("P3.5", [6 / 1.5 ** 2, fdist.sf(6 / 1.5 ** 2, 2, 15)])                   # P3.5: MSE × 2.25
pv = np.array([0.004, 0.012, 0.021, 0.034, 0.18, 0.47])                                 # P3.6: Bonferroni
assert agrees("P3.6", [(pv <= 0.05).sum(), (pv <= 0.05 / 6).sum()])

claims = pd.DataFrame([[13.5, 13, 15.5], [14.5, 17, 19.5], [18.5, 21, 20.5], [22.5, 23, 23.5], [23.5, 26, 28.5]],
                      index=[f"handler {k}" for k in range(1, 6)], columns=["tool 1", "tool 2", "tool 3"])
long = claims.stack().rename("minutes").rename_axis(["handler", "tool"]).reset_index()
grand = long["minutes"].mean()
sst = ((long["minutes"] - grand) ** 2).sum()
sstr = 5 * ((long.groupby("tool")["minutes"].mean() - grand) ** 2).sum()
ssbl = 3 * ((long.groupby("handler")["minutes"].mean() - grand) ** 2).sum()
sse = sst - sstr - ssbl
p_crd = fdist.sf((sstr / 2) / ((sst - sstr) / 12), 2, 12)                               # ignoring the handlers
p_blk = fdist.sf((sstr / 2) / (sse / 8), 2, 8)                                          # handlers as blocks
print(f"P3.7 SSTR {sstr:g}, SSBL {ssbl:g}, SSE {sse:g}: p {p_crd:.2f} ignoring the handlers, {p_blk:.4f} with blocks")
assert agrees("P3.7", [p_crd, p_blk])

pack = pd.DataFrame({"packaging": ["standard"] * 6 + ["eco"] * 6,
                     "shipping": (["standard"] * 3 + ["express"] * 3) * 2,
                     "rating": [7.6, 7.9, 7.3, 8.4, 8.1, 8.7, 7.8, 7.4, 7.7, 9.6, 9.9, 9.3]})
g = pack["rating"].mean()
ssa = 6 * ((pack.groupby("packaging")["rating"].mean() - g) ** 2).sum()
ssb = 6 * ((pack.groupby("shipping")["rating"].mean() - g) ** 2).sum()
cells = pack.groupby(["packaging", "shipping"])["rating"]
sse = ((pack["rating"] - cells.transform("mean")) ** 2).sum()
ssab = ((pack["rating"] - g) ** 2).sum() - ssa - ssb - sse
F_ab = ssab / (sse / 8)
print(cells.mean().unstack().round(2).to_string())
print(f"P3.8 SSAB {ssab:.3f}, SSE {sse:.3f}: F {F_ab:.2f}, p {fdist.sf(F_ab, 1, 8):.4f}")
assert agrees("P3.8", [F_ab, fdist.sf(F_ab, 1, 8)])

## Tier 4 · The case: Tarnwick Components' resin suppliers

Tarnwick Components moulds plastic housings for power tools (a fictional company; our own data). It must standardise
on one of four resin suppliers: Alder, Birch, Cedar and Dunmore. A trial moulded 24 batches from each, in random
order. For each batch it measured the tensile strength of the test bars and counted the parts rejected at inspection,
out of 500. The plan, written before the trial: every batch at least 62 MPa, a process spread σ of at most 3 MPa,
α = 0.05, and Bonferroni when comparing pairs. Resin costs $2.35, $2.20, $2.05 and $2.10 a kilogram, and the plant
uses 400 tonnes a year.

| column | what it is |
|---|---|
| `batch` | batch id, in moulding order, B01–B96 |
| `supplier` | Alder, Birch, Cedar or Dunmore |
| `strength_mpa` | tensile strength of the batch's test bars (MPa) |
| `rejects` | parts rejected at inspection, out of 500 |

The next cell builds the table from a seeded random generator, so everyone gets exactly the same 96 batches (the same
as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
CASE_SEED = 7
SUPPLIERS = ["Alder", "Birch", "Cedar", "Dunmore"]
PLAN = {"Alder": (68.0, 2.4, 0.020), "Birch": (66.4, 2.5, 0.021), "Cedar": (69.0, 5.0, 0.034),
        "Dunmore": (64.4, 2.5, 0.023)}           # planned mean strength (MPa), spread, reject rate
PARTS_PER_BATCH, SPEC_MPA, SPEC_SIGMA, TONNES_YEAR = 500, 62, 3, 400
PRICE_KG = {"Alder": 2.35, "Birch": 2.20, "Cedar": 2.05, "Dunmore": 2.10}

def build_case(seed=CASE_SEED):
    """24 batches a supplier: strengths standardised to the planned mean and spread, rejects as 500 trials; shuffled."""
    r = random.Random(seed); rows = []
    for name in SUPPLIERS:
        mu, s_, p = PLAN[name]
        z = [r.gauss(0, 1) for _ in range(24)]
        m = sum(z) / 24; sz = math.sqrt(sum((v - m) ** 2 for v in z) / 23)
        for v in z:
            rows.append((name, round(mu + s_ * (v - m) / sz, 1), sum(1 for _ in range(PARTS_PER_BATCH) if r.random() < p)))
    r.shuffle(rows)
    return pd.DataFrame({"batch": [f"B{k + 1:02d}" for k in range(len(rows))], "supplier": [x[0] for x in rows],
                         "strength_mpa": [x[1] for x in rows], "rejects": [x[2] for x in rows]})

df = build_case()

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'n': {'Alder': 24, 'Birch': 24, 'Cedar': 24, 'Dunmore': 24},
 'mean': {'Alder': 67.99583333333334,
          'Birch': 66.39999999999999,
          'Cedar': 69.00833333333334,
          'Dunmore': 64.39999999999999},
 'sd': {'Alder': 2.400267950018092,
        'Birch': 2.502172968684898,
        'Cedar': 4.9969483441019475,
        'Dunmore': 2.5028679202241095},
 'low': {'Alder': 0, 'Birch': 0, 'Cedar': 2, 'Dunmore': 3},
 'rej': {'Alder': 256, 'Birch': 256, 'Cedar': 400, 'Dunmore': 275},
 'rej_chi': 49.94529939884678,
 'rej_p': 8.206380019970538e-11,
 'var_chi_cedar': 63.81092592592594,
 'anova3_F': 12.781033568948434,
 'anova3_p': 1.8969618690412027e-05,
 'anova4_F': 8.977621711435173,
 'var_ratio': 4.334013577634326,
 'below_normal': 0.08037961285737212,
 'saving_year': 59999.99999999996}
# @END STATS
S = CASE_STATS
by = df.groupby("supplier")
assert len(df) == 96 and by.size().to_dict() == S["n"]
assert all(abs(by["strength_mpa"].mean()[k] - S["mean"][k]) < 1e-9 for k in SUPPLIERS)
assert by["rejects"].sum().to_dict() == S["rej"]
print(f"{len(df)} batches, {df.shape[1]} columns: the same data as the course's case.")
df.head(8)

### How you'd do this in Python

With the batches in a DataFrame, each step of the case is a line or two: `groupby("supplier")` for the means and
spreads, `stats.chi2_contingency` for the reject rates, `chi2.sf` for each supplier's spread against the 3 MPa
limit, and `stats.f_oneway` for the means. The cells below reproduce the video's case: the tempting pick (the
strongest mean), the spread that rules it out, and the comparison of the three that remain.

In [ ]:
summary = by.agg(n=("strength_mpa", "size"), mean=("strength_mpa", "mean"), s=("strength_mpa", "std"),
                 below_spec=("strength_mpa", lambda x: (x < SPEC_MPA).sum()), rejects=("rejects", "sum"))
summary["reject rate"] = summary["rejects"] / (summary["n"] * PARTS_PER_BATCH)
summary["χ² for σ ≤ 3"] = (summary["n"] - 1) * summary["s"] ** 2 / SPEC_SIGMA ** 2
summary["p"] = chi2.sf(summary["χ² for σ ≤ 3"], summary["n"] - 1)
print(summary.round(4).to_string(), "\n")

rej = summary["rejects"].to_numpy(); parts = (summary["n"] * PARTS_PER_BATCH).to_numpy()
r = stats.chi2_contingency([rej, parts - rej])
print(f"reject rates equal? χ² = {r.statistic:.2f}, df {r.dof}, p = {r.pvalue:.1e}")
three = stats.f_oneway(*(df.loc[df["supplier"] == k, "strength_mpa"] for k in ("Alder", "Birch", "Dunmore")))
print(f"ANOVA, Alder · Birch · Dunmore: F = {three.statistic:.2f}, p = {three.pvalue:.5f}")
assert round(r.statistic, 2) == 49.95 and round(summary.loc["Cedar", "χ² for σ ≤ 3"], 1) == 63.8
assert round(three.statistic, 2) == 12.78

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.15)
rng = np.random.default_rng(1)
for k, name in enumerate(SUPPLIERS):
    x = df.loc[df["supplier"] == name, "strength_mpa"]
    ax.scatter(x, k + rng.uniform(-0.18, 0.18, len(x)), s=22, color=GROUPS[k], alpha=0.8)
    ax.plot([x.mean()] * 2, [k - 0.3, k + 0.3], color=GROUPS[k], lw=3)
    ax.text(x.max() + 0.6, k, f"mean {x.mean():.1f} · s {x.std():.2f}", color=GROUPS[k], va="center", fontsize=12)
ax.axvline(SPEC_MPA, color=CORAL, ls="--", lw=1.6); ax.text(SPEC_MPA + 0.3, 3.55, "spec 62 MPa", color=CORAL)
ax.set_yticks(range(4), SUPPLIERS); ax.set_ylim(-0.6, 3.9); ax.set_xlim(54, 90); ax.set_xlabel("tensile strength (MPa)")
fig.suptitle("Cedar is strongest on average, and by far the least consistent", x=0.1, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the ANOVA with all four suppliers (and why it misleads), the
share of Cedar batches a normal model expects below spec, and the money, ending in your recommendation. Use the
DataFrame `df` in a cell of your own (press **+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `df` with pandas and scipy and checks it
against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
grp = {k: df.loc[df["supplier"] == k, "strength_mpa"].to_numpy() for k in SUPPLIERS}
mean, sd = {k: v.mean() for k, v in grp.items()}, {k: v.std(ddof=1) for k, v in grp.items()}
mine = {}
mine["P4.1"] = [mean[k] for k in SUPPLIERS] + [sd[k] for k in SUPPLIERS]
r = stats.chi2_contingency([rej, parts - rej])
mine["P4.2"] = [rej.sum(), r.statistic, r.pvalue]
mar = marascuilo(rej, parts, SUPPLIERS)
mine["P4.3"] = [", ".join(pair for pair, *_, differs in mar if differs)]
x2 = {k: 23 * sd[k] ** 2 / SPEC_SIGMA ** 2 for k in SUPPLIERS}
fail = ", ".join(k for k in SUPPLIERS if chi2.sf(x2[k], 23) <= 0.05)
mine["P4.4"] = [x2["Cedar"], chi2.sf(x2["Cedar"], 23), fail]
a3 = stats.f_oneway(grp["Alder"], grp["Birch"], grp["Dunmore"])
mine["P4.5"] = [a3.statistic, 2, 69, a3.pvalue]
mse3 = sum(23 * sd[k] ** 2 for k in ("Alder", "Birch", "Dunmore")) / 69
lsd_p = {(a, b): 2 * stats.t.sf(abs(mean[a] - mean[b]) / math.sqrt(mse3 * (2 / 24)), 69)
         for a, b in (("Alder", "Birch"), ("Alder", "Dunmore"), ("Birch", "Dunmore"))}
survive = ", ".join(f"{a}–{b}" for (a, b), p in lsd_p.items() if p <= 0.05 / 3)
mine["P4.6"] = list(lsd_p.values()) + [survive]
a4 = stats.f_oneway(*grp.values())
mine["P4.7"] = [a4.statistic, a4.pvalue, sd["Cedar"] ** 2 / sd["Alder"] ** 2]
saving = (PRICE_KG["Alder"] - PRICE_KG["Birch"]) * TONNES_YEAR * 1000
below = stats.norm.cdf((SPEC_MPA - mean["Cedar"]) / sd["Cedar"])
mine["P4.8"] = [saving, below]

for k, got in mine.items():
    got = [g if isinstance(g, str) else float(g) for g in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(g if isinstance(g, str) else round(g, 6))}" for p, g in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")
f_ca = 2 * fdist.sf(sd["Cedar"] ** 2 / sd["Alder"] ** 2, 23, 23)
print(f"Cedar's spread against Alder's: F = {sd['Cedar'] ** 2 / sd['Alder'] ** 2:.2f}, two-tailed p = {f_ca:.4f}. "
      f"A normal model puts {100 * below:.0f}% of Cedar batches below spec: about {TONNES_YEAR * below:.0f} tonnes a year.")
assert round(f_ca, 4) == 0.0008 and round(below, 3) == 0.080

## Sources
Every exercise, data set and company here is our own (Tarnwick Components is fictional). The chapter order follows a
standard first course in business statistics: inferences about population variances, tests of goodness of fit,
independence and multiple proportions, then experimental design and analysis of variance.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs05_chi_square_anova/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs05_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert all((a == b) if isinstance(b, str) else abs(a - float(b)) < 1e-9
                   for a, b in zip(_q["answer"], _flat(_e["answer"]))) and len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    assert _P.CASE["strength_mpa"] == df["strength_mpa"].tolist() and _P.CASE["rejects"] == df["rejects"].tolist()
    assert _P.CASE["supplier"] == df["supplier"].tolist() and _P.CASE["batch"] == df["batch"].tolist()
    print("matches the course's practice set")